In [ ]:
import pandas as pd
from itertools import product, chain
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [ ]:
# Read Data
dmu_io=pd.read_csv('Data/Model Data New.csv')
dmu_io['ServiceTypeLAB']=['P2P' if x=='Point-to-Point' else
                          'FFR' if x=='Flexible Fixed-Route' else
                          'FR'
                          for x in dmu_io.ServiceType]

# Decision Making Unit
dmus=list(dmu_io.RouteID)


# Metrics of each service intention
SI_df=pd.read_csv('SI_Metric.csv')
ope_input=['FleetNum','MonthlyTrips','MonthlyVKT','KMCost']
ope_output=['MonthlyPatronage','Util_Rate']

si_output={}
for _, row in SI_df.iterrows():
    intention=row['ServiceIntention']
    categories=row[row==1].index.tolist()
    si_output[intention]=[x for x in categories]
#     si_output[intention]=[x+'_NM' for x in categories]
all_si=np.array(SI_df.ServiceIntention)


# Input & Output Indicators used for each DMU
inattr=dict(zip(dmus, [ope_input for y in range(len(dmu_io))]))
outattr=dict(zip(dmus, [np.unique(list(chain.from_iterable([si_output[x]+ope_output for x in all_si[np.array(dmu_io[all_si])[y]==1]]))) for y in range(len(dmu_io))]))
inattr_all=np.unique([item for sublist in list(inattr.values()) for item in sublist])
outattr_all=np.unique([item for sublist in list(outattr.values()) for item in sublist])


# All Input & Output
inputs=dict(zip(dmus, dmu_io.loc[:,inattr_all].to_dict(orient='records')))
outputs=dict(zip(dmus, dmu_io.loc[:,outattr_all].to_dict(orient='records')))


# Operational Type (Fixed-Route. Flexible Fixed-Route, Point-to-Point )
dmus_group=dict(zip(dmu_io['RouteID'], dmu_io['ServiceTypeLAB']))

# Policy-oriented Meta-frontier DDF

In [ ]:
def dff_model(dmus, inattr, outattr, inputs, outputs):
    dff=dict()

    for target in dmus:
        model=gp.Model('DEA')
        model.params.OutputFlag=0 # mute the model output messages

        beta=model.addVar(lb=0, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name='beta')

        lada={}
        for i in dmus:
            lada[i]=model.addVar(lb=0, ub=GRB.INFINITY, vtype=GRB.CONTINUOUS, name=i)


        # Objective
        model.setObjective(beta, GRB.MAXIMIZE)


        # set g_x and g_y
        g_x=inputs[target].copy()
        for x in inattr_all:
            if x not in inattr[target]:
                g_x[x]=0

        g_y=outputs[target].copy()
        for x in outattr_all:
            if x not in outattr[target]:
                g_y[x]=0


        # Constraints 1
        for i in inattr_all:
            s='input_'+i
            model.addConstr(sum(np.array([lada[x] for x in dmus])*np.array([inputs[x][i] for x in dmus]))-inputs[target][i]+beta*g_x[i]<=0, name=s)

        # Constraints 2
        for i in outattr_all:
            s='output_'+i
            model.addConstr(sum(np.array([lada[x] for x in dmus])*np.array([outputs[x][i] for x in dmus]))-outputs[target][i]-beta*g_y[i]>=0, name=s)

        # Constraints 3 (the wieghted sum lambda is 1)
        model.addConstr(sum(np.array([lada[x] for x in dmus]))==1, name=s)


        model.optimize()

        temp=dict({'beta': model.objVal,
                   'lambda': [lada[x].getAttr(GRB.Attr.X) for x in dmus],
                   'g_x':g_x,
                   'g_y':g_y})

        dff[target]=temp
    
    return(dff)


def meta_dff_model(dmus, dmus_group, inattr, outattr, inputs, outputs):
    dff_meta=dff_model(dmus, inattr, outattr, inputs, outputs)
    
    all_group=list(np.unique(np.array([dmus_group[x] for x in dmus_group])))
    dff_group=dict()
    dff_group_eff=dict()
    
    for i in all_group:
        temp_group=[x for x in dmus_group if dmus_group[x]==i]
        dff_res=dff_model(temp_group, inattr, outattr,
                          {x: inputs[x] for x in temp_group}, 
                          {x: outputs[x] for x in temp_group})
        dff_group[i]=dff_res
        dff_group_eff=dff_group_eff | {x: 1-dff_res[x]['beta'] for x in temp_group}
    
    return(dict({'dff_meta': dff_meta, 'dff_group': dff_group, 'dff_group_eff':dff_group_eff}))

In [ ]:
model_res=meta_dff_model(dmus, dmus_group, inattr, outattr, inputs, outputs)

In [ ]:
mf_ddf=pd.merge(pd.DataFrame.from_dict({'GroupEff': model_res['dff_group_eff']}),
                pd.DataFrame.from_dict({'MetaEff': {x: 1-model_res['dff_meta'][x]['beta'] for x in dmus}}),
                how='left', left_index=True, right_index=True)
mf_ddf=pd.merge(mf_ddf, pd.DataFrame.from_dict({'ClusterName': dmus_group}), how='left', left_index=True, right_index=True)
mf_ddf['TGR']=mf_ddf['MetaEff']/mf_ddf['GroupEff']
mf_ddf['DMU']=mf_ddf.index
mf_ddf.loc[:,['DMU','ClusterName','MetaEff','GroupEff','TGR']]
# mf_ddf.to_csv('MetaDDF_new.csv', index=False)

In [ ]:
# # Read Data
# dmu_io=pd.read_csv('Data/Model Data New.csv')
# dmu_io=dmu_io[dmu_io.RouteID.str.contains('ILA_09').tolist()].reset_index(drop=True)

# # Decision Making Unit
# dmus=list(dmu_io.RouteID)


# # Metrics of each service intention
# SI_df=pd.read_csv('SI_Metric.csv')
# ope_input=['FleetNum','MonthlyTrips','MonthlyVKT','KMCost']
# ope_output=['MonthlyPatronage','Util_Rate']

# si_output={}
# for _, row in SI_df.iterrows():
#     intention=row['ServiceIntention']
#     categories=row[row==1].index.tolist()
#     si_output[intention]=[x for x in categories]
# #     si_output[intention]=[x+'_NM' for x in categories]
# all_si=np.array(SI_df.ServiceIntention)


# # Input & Output Indicators used for each DMU
# inattr=dict(zip(dmus, [ope_input for y in range(len(dmu_io))]))
# outattr=dict(zip(dmus, [np.unique(list(chain.from_iterable([si_output[x]+ope_output for x in all_si[np.array(dmu_io[all_si])[y]==1]]))) for y in range(len(dmu_io))]))
# inattr_all=np.unique([item for sublist in list(inattr.values()) for item in sublist])
# outattr_all=np.unique([item for sublist in list(outattr.values()) for item in sublist])


# # All Input & Output
# inputs=dict(zip(dmus, dmu_io.loc[:,inattr_all].to_dict(orient='records')))
# outputs=dict(zip(dmus, dmu_io.loc[:,outattr_all].to_dict(orient='records')))

# dff_model(dmus, inattr, outattr, inputs, outputs)

